# Step 5 — NammaMart ETL pipeline test

This notebook runs the same ETL implementation used by the Airflow DAG. It is intentionally local and repeatable so the star schema, quarantine, reconciliation and reports can be inspected before starting Docker/Airflow.

Flow: extract → validate → quarantine → transform → load DuckDB → reconcile payments → build reports.

In [1]:
from pathlib import Path
import sys
import duckdb

# Import the exact module that the Airflow DAG will use.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_etl import run_etl

# Use an assignment-specific warehouse and quarantine directory so this test
# does not overwrite the original Meridian demo warehouse.
warehouse = Path('assignment/warehouse_samy_step5.duckdb')
quarantine = Path('assignment/quarantine_samy_step5')
result = run_etl(
    warehouse_path=warehouse,
    quarantine_dir=quarantine,
    run_id='step5-local-test',
)

print('ETL run completed')
print(f"Rows extracted:   {result['rows_extracted']:,}")
print(f"Rows loaded:      {result['rows_loaded']:,}")
print(f"Rows quarantined: {result['rows_quarantined']:,}")
print(f"Reconciliation rows: {result['reconciliation_rows']:,}")
print(f"Warehouse: {result['warehouse_path']}")

ETL run completed
Rows extracted:   5,212
Rows loaded:      1,903
Rows quarantined: 497
Reconciliation rows: 629
Warehouse: assignment/warehouse_samy_step5.duckdb


In [2]:
# Inspect the tables created by the idempotent DuckDB load.
con = duckdb.connect(str(warehouse), read_only=True)
table_counts = con.execute("""
        SELECT table_name, estimated_size
        FROM duckdb_tables()
        WHERE schema_name = 'main'
        ORDER BY table_name
    """).fetchdf()
display(table_counts)

# The fact-table count is the finance-safe count after critical failures
# have been excluded and dimension joins have enforced referential integrity.
display(con.execute('SELECT COUNT(*) AS fact_rows FROM etl_fact_orders').fetchdf())
display(con.execute('SELECT * FROM etl_daily_zone_revenue ORDER BY order_date, zone, store_id LIMIT 10').fetchdf())
display(con.execute('SELECT * FROM etl_category_margin ORDER BY category').fetchdf())
con.close()

,table_name,estimated_size
0,dq_results,38
1,etl_category_margin,6
2,etl_daily_zone_revenue,384
3,etl_dim_customer,400
4,etl_dim_customer_history,400
5,etl_dim_product,142
6,etl_dim_store,13
7,etl_fact_orders,1903
8,etl_payment_reconciliation,629
9,pipeline_runs,1


,fact_rows
0,1903


,order_date,zone,store_id,daily_net_revenue
0,2026-09-01,East,S08,2984.70
1,2026-09-01,East,S09,2089.60
2,2026-09-01,East,S10,2141.60
3,2026-09-01,East,S11,2121.15
4,2026-09-01,North,S01,1410.80
5,2026-09-01,North,S02,2385.55
6,2026-09-01,North,S03,4583.80
7,2026-09-01,South,S05,803.00
8,2026-09-01,South,S06,2812.00
9,2026-09-01,West,S13,1469.10


,category,revenue,gross_margin
0,Dairy & Bakery,149028.75,39177.96
1,Fruits & Vegetables,75956.15,18226.14
2,Household,166866.55,42326.26
3,Personal Care,179965.60,39824.12
4,Snacks & Beverages,148381.10,33965.22
5,Staples,212686.00,54563.95


In [3]:
# Demonstrate idempotency by running the same source data a second time.
second = run_etl(
    warehouse_path=warehouse,
    quarantine_dir=quarantine,
    run_id='step5-local-rerun',
)
con = duckdb.connect(str(warehouse), read_only=True)
fact_count_after_rerun = con.execute('SELECT COUNT(*) FROM etl_fact_orders').fetchone()[0]
con.close()
print(f"Fact rows after rerun: {fact_count_after_rerun:,}")
print('The table is replaced from the same source, so revenue is not double-counted.')

Fact rows after rerun: 1,903
The table is replaced from the same source, so revenue is not double-counted.
